# SWIFT message flow - guide-based Sankey sketch

A separate exploratory sketch based on the supplied **17-step processor diagram** and its eight-part guide. The original `20261003_sankey_test.ipynb` is preserved.

**Read ribbon width as a synthetic message/event count, never money.** This is an illustration of the supplied architecture, not a verified description of current Swift infrastructure. Regional and slice processor names are retained from the reference; New York/London are not asserted as actual routing locations. `MON` is retained as a source label without interpreting it as compliance screening.

Three panels follow the same cohort:
1. **Message path:** submission, validation/storage, transmission, receiving-side processing and bank storage.
2. **Sender acknowledgement:** storage confirmation and ACK/NAK back to Bank A.
3. **Delivery feedback and reports:** UAK/UNK, report storage, report copy and sender-side report storage.

The panels have independent scales. The same cohort appears in all three; do not add their totals. Repeated stages carry the same messages, not new payments. Processor nodes that appear again represent a later event at the same component. Storage operations are attached to their owning processing stage rather than drawn as an extra copy of the payment.

## Editable scenario

Start with 100 submitted instructions: 4 fail sender-side technical validation, 96 are accepted by the network, and 2 of those receive a negative acknowledgement at the receiving interface. The remaining 94 are stored successfully by Bank B.

These rates are invented for visibility, not Swift performance statistics. For clarity, this sketch treats an UNK as an unsuccessful first delivery attempt and stops there; it does not model retries, recovery, expiry or subsequent business rejection. An ACK/UAK does **not** establish settlement or beneficiary credit. Report generation is simplified to one report per receiving-side response.

In [1]:
from pathlib import Path
from collections import Counter
import plotly.graph_objects as go

submitted = 100
network_rejected = 4
receiver_negative = 2

assert all(isinstance(v, int) for v in (submitted, network_rejected, receiver_negative))
assert submitted > 0
assert 0 <= network_rejected < submitted
network_accepted = submitted - network_rejected
assert 0 <= receiver_negative <= network_accepted
receiver_accepted = network_accepted - receiver_negative

print(f"Submitted: {submitted} | Network accepted: {network_accepted} | NAK: {network_rejected}")
print(f"Receiver UAK: {receiver_accepted} | Receiver UNK: {receiver_negative}")

Submitted: 100 | Network accepted: 96 | NAK: 4
Receiver UAK: 94 | Receiver UNK: 2


## How the guide maps to the chart

| Reference step | Panel and representation |
|---|---|
| 1 | Message path: Bank A sends to Regional Processor A |
| 2 | Message path: Regional Processor A sends to Slice Processor A |
| 3 | Message path: accepted messages stored at Slice Processor A |
| 4 | Sender acknowledgement: storage confirmation sent to Regional Processor A |
| 5 | Sender acknowledgement: ACK or NAK returned to Bank A |
| 6 | Message path: accepted message sent to Regional Processor B |
| 7 | Message path: temporary storage at Regional Processor B |
| 8-9 | Message path: Slice Processor B receives message and validates MON (source label) |
| 10 | Message path: Regional Processor B authorised to deliver |
| 11 | Message path: delivery attempt to Bank B |
| 12 | Message path: successful receipt stored in Bank B systems |
| 13 | Delivery feedback: UAK or UNK returned by Bank B |
| 14 | Delivery feedback: report sent to Slice Processor B |
| 15 | Delivery feedback: report stored on receiving side |
| 16 | Delivery feedback: report copy sent to Slice Processor A |
| 17 | Delivery feedback: report stored on sending side |

The red NAK/UNK branches make negative outcomes explicit; the source picture mainly shows the successful path. The step 10 authorisation is grouped with Regional Processor B's delivery action. Security, resilience, audit and compliance are contextual controls, not additional quantities flowing through the Sankey.

In [2]:
COLORS = {
    "bank": "#5e9bea", "processor": "#43c7c4", "storage": "#78cd9a",
    "ack": "#e7bc65", "report": "#b297ec", "negative": "#ee7c88",
}

def rgba(hex_color, alpha=0.38):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

# Node: (ID, displayed label, colour role, x, y, hover explanation)
# Link: (source ID, target ID, count, colour role, hover explanation)
message_nodes = [
    ("a", "Bank A", "bank", .01, .32, "Starting cohort of payment instructions; no claim of settlement."),
    ("ra", "Regional A", "processor", .14, .32, "Step 1: receives Bank A's instruction."),
    ("sa", "Slice A<br>validate + store", "storage", .29, .32, "Steps 2-3: technical validation; accepted messages are stored."),
    ("nak", "Network rejected", "negative", .47, .88, "Sender-side validation failure. NAK shown separately in panel 2."),
    ("rb", "Regional B<br>temporary store", "storage", .47, .32, "Steps 6-7: cross-region transmission and temporary storage."),
    ("sb", "Slice B<br>validate MON", "processor", .64, .32, "Steps 8-9: source diagram's MON validation; not assumed to be sanctions screening."),
    ("delivery", "Regional B<br>authorised delivery", "processor", .80, .32, "Step 10: authorised to send to Bank B."),
    ("stored", "Bank B<br>received + stored", "bank", .99, .29, "Steps 11-12: successful technical receipt and bank storage, not beneficiary credit."),
    ("unk", "Bank B<br>negative receipt", "negative", .99, .88, "Step 11 attempt receives UNK; no successful storage asserted. Retries omitted."),
]
message_links = [
    ("a", "ra", submitted, "processor", "1 - Send message"),
    ("ra", "sa", submitted, "processor", "2 - Send to Slice Processor A"),
    ("sa", "nak", network_rejected, "negative", "Technical validation failed; stop this attempt"),
    ("sa", "rb", network_accepted, "storage", "3, 6-7 - Stored, transmitted and temporarily stored"),
    ("rb", "sb", network_accepted, "processor", "8-9 - Send to Slice B / validate MON"),
    ("sb", "delivery", network_accepted, "processor", "10 - Authorise Regional B to deliver"),
    ("delivery", "stored", receiver_accepted, "storage", "11-12 - Successful receipt and storage"),
    ("delivery", "unk", receiver_negative, "negative", "11 - Negative receiving-interface acknowledgement"),
]

ack_nodes = [
    ("stored_a", "Slice A<br>storage confirmed", "storage", .01, .23, "Step 3: accepted messages have been stored."),
    ("regional_ack", "Regional A<br>storage ACK", "ack", .48, .23, "Step 4: receives storage confirmation."),
    ("bank_ack", "Bank A<br>ACK received", "ack", .99, .23, "Step 5: network acceptance, not payment completion."),
    ("validation_fail", "Validation<br>failed", "negative", .01, .83, "Synthetic technical rejection outcome."),
    ("regional_nak", "Regional A<br>NAK path", "negative", .48, .83, "Rejection response returned through the sender-side interface."),
    ("bank_nak", "Bank A<br>NAK received", "negative", .99, .83, "Step 5: message rejected; correction/retry is outside this sketch."),
]
ack_links = [
    ("stored_a", "regional_ack", network_accepted, "ack", "4 - Confirm stored message"),
    ("regional_ack", "bank_ack", network_accepted, "ack", "5 - Send ACK to Bank A"),
    ("validation_fail", "regional_nak", network_rejected, "negative", "Return validation failure"),
    ("regional_nak", "bank_nak", network_rejected, "negative", "5 - Send NAK to Bank A"),
]

report_nodes = [
    ("bank_b", "Bank B<br>delivery responses", "bank", .01, .35, "Responses to the accepted cohort's delivery attempts."),
    ("uak", "UAK<br>positive receipt", "ack", .20, .25, "Step 13: positive user acknowledgement; not beneficiary-credit confirmation."),
    ("unk", "UNK<br>negative receipt", "negative", .20, .85, "Step 13: negative user acknowledgement."),
    ("rb_report", "Regional B<br>collect responses", "report", .40, .35, "Steps 13-14: receipt responses and report transmission."),
    ("sb_report", "Slice B<br>store report", "report", .60, .35, "Step 15: receiving-side report stored (source graphic labels its cylinder Messages)."),
    ("sa_report", "Slice A<br>receive report copy", "report", .80, .35, "Step 16: copy sent back to Slice Processor A."),
    ("archive", "Sender-side<br>report storage", "storage", .99, .35, "Step 17: report stored for this technical delivery outcome."),
]
report_links = [
    ("bank_b", "uak", receiver_accepted, "ack", "13 - Send UAK"),
    ("bank_b", "unk", receiver_negative, "negative", "13 - Send UNK"),
    ("uak", "rb_report", receiver_accepted, "report", "13 - Positive response received by Regional B"),
    ("unk", "rb_report", receiver_negative, "negative", "13 - Negative response received by Regional B"),
    ("rb_report", "sb_report", network_accepted, "report", "14-15 - Send and store report"),
    ("sb_report", "sa_report", network_accepted, "report", "16 - Send a copy of the report"),
    ("sa_report", "archive", network_accepted, "report", "17 - Store the report"),
]

In [3]:
def sankey_trace(name, nodes, links, domain_y):
    # Omit zero-width paths and unused nodes when the scenario is edited.
    links = [link for link in links if link[2] > 0]
    active = {endpoint for link in links for endpoint in link[:2]}
    nodes = [node for node in nodes if node[0] in active]
    index = {node[0]: i for i, node in enumerate(nodes)}
    return go.Sankey(
        name=name, arrangement="fixed", domain=dict(x=[.025, .97], y=domain_y),
        node=dict(
            label=[n[1] for n in nodes], color=[COLORS[n[2]] for n in nodes],
            x=[n[3] for n in nodes], y=[n[4] for n in nodes],
            customdata=[n[5] for n in nodes], pad=24, thickness=15,
            line=dict(color="rgba(255,255,255,.3)", width=.6),
            hovertemplate="<b>%{label}</b><br>%{customdata}<extra></extra>",
        ),
        link=dict(
            source=[index[e[0]] for e in links], target=[index[e[1]] for e in links],
            value=[e[2] for e in links], color=[rgba(COLORS[e[3]]) for e in links],
            customdata=[e[4] for e in links],
            hovertemplate="<b>%{customdata}</b><br>%{value:,.0f} synthetic messages / events<extra></extra>",
        ),
        textfont=dict(size=12, color="#edf3fc"),
    )

fig = go.Figure([
    sankey_trace("Message path", message_nodes, message_links, [.69, .96]),
    sankey_trace("Sender acknowledgement", ack_nodes, ack_links, [.40, .56]),
    sankey_trace("Delivery feedback and reports", report_nodes, report_links, [.04, .27]),
])
fig.update_layout(
    title=dict(text="SWIFT message flow | guide-based sketch", x=.03, font=dict(size=25)),
    template="plotly_dark", paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(family="Arial, sans-serif", size=12, color="#edf3fc"),
    height=1280, autosize=True, margin=dict(l=45, r=65, t=160, b=70),
    annotations=[
        dict(x=.025, y=1.095, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text=f"Synthetic cohort: {submitted} submitted | {network_accepted} network accepted | {receiver_accepted} positive receipts", font=dict(size=14, color="#b7c6db")),
        dict(x=.025, y=1.055, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text="Width = message / event count. Independent panel scales. Technical receipt does not establish settlement.", font=dict(size=12, color="#b7c6db")),
        dict(x=.025, y=1.005, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text="<b>01 / MESSAGE PATH</b>  -  validation, storage and delivery", font=dict(size=16)),
        dict(x=.025, y=.61, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text="<b>02 / ACK / NAK TO BANK A</b>  -  network acceptance or rejection", font=dict(size=16)),
        dict(x=.025, y=.315, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text="<b>03 / DELIVERY FEEDBACK + REPORTS</b>  -  read left to right; reports travel back to the sender", font=dict(size=16)),
        dict(x=.025, y=-.04, xref="paper", yref="paper", xanchor="left", showarrow=False,
             text="Based on the supplied processor diagram; illustrative topology and counts. Hover ribbons for reference step numbers.", font=dict(size=12, color="#93a6c0")),
    ],
)
fig.show(renderer="plotly_mimetype")

## Count checks and export

Checks below validate the illustrative cohort and conservation within each panel. They do not validate the supplied processor architecture. The export includes Plotly locally inside the HTML so it can be opened offline; a wide canvas preserves stage labels on smaller screens.

In [4]:
for name, nodes, links in [
    ("message", message_nodes, message_links),
    ("acknowledgement", ack_nodes, ack_links),
    ("reports", report_nodes, report_links),
]:
    ids = {node[0] for node in nodes}
    assert len(ids) == len(nodes), f"Duplicate node in {name}"
    incoming, outgoing = Counter(), Counter()
    for source, target, value, role, detail in links:
        assert source in ids and target in ids and value >= 0
        outgoing[source] += value
        incoming[target] += value
    for node in incoming.keys() & outgoing.keys():
        assert incoming[node] == outgoing[node], f"Unbalanced node: {name}/{node}"
assert submitted == network_rejected + receiver_accepted + receiver_negative
print("All cohort and intermediate-node count checks passed.")

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "20261009_swift_guide_sankey.html"
plot_html = fig.to_html(full_html=False, include_plotlyjs=True,
                        config={"responsive": True, "displaylogo": False})
html = """<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SWIFT guide-based Sankey sketch</title>
<style>
body { margin: 0; background: #0d1117; color: #edf3fc; font-family: Arial, sans-serif; }
main { padding: 20px; }
.intro { max-width: 1050px; line-height: 1.6; color: #b7c6db; }
a { color: #9bc6ff; }
.scroll { overflow-x: auto; }
.chart { min-width: 1600px; }
</style></head><body><main>
<a href="../index.html">Back to phey.app</a>
<p class="intro">A sketch of the supplied Swift processor guide. All counts are synthetic.
The three panels follow the same cohort with independent scales: do not add their totals.
ACK/UAK indicates technical acceptance or receipt, not beneficiary credit.
Processor topology is illustrative. Hover ribbons for step numbers; scroll horizontally on smaller screens.</p>
<div class="scroll"><div class="chart">""" + plot_html + """</div></div>
</main></body></html>"""
output_file.write_text(html, encoding="utf-8")
print(f"Exported: {output_file.resolve()}")

All cohort and intermediate-node count checks passed.
Exported: C:\Users\PH\Python\p00\output\20261009_swift_guide_sankey.html


## Scope and reference notes

- This notebook follows the **user-supplied guide and picture**, including its legacy FIN acknowledgement terminology. It is not a simulation of current FINplus internals.
- For a current production payment model, distinguish ISO 20022 message exchange, bank business processing, settlement and beneficiary credit. Swift's [implementation FAQ](https://www.swift.com/standards/iso-20022/iso-20022-faqs/implementation) describes the end of CBPR+ coexistence on 22 November 2025.
- [Universal Confirmations](https://www.swift.com/payments/universal-confirmations) describes the separate payment-outcome confirmation layer.
- [Swift's RMA explanation](https://www.swift.com/fr/node/34546) describes counterparty messaging permissions, distinct from cryptographic authentication.
- [Swift and sanctions](https://www.swift.com/about-us/legal/compliance-0/swift-and-sanctions) explains the institutions' compliance responsibilities. The chart does not invent a universal sanctions check inside a slice processor.

To explore a clean success path, set `network_rejected = 0` and `receiver_negative = 0`, then run all cells. Zero-width negative paths will be hidden automatically.